# NLP Masterclass 05: Pre-training Objectives & BERT

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 04 (Seq2Seq & Attention), DL 07 (Transformer Architecture)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"In Word2Vec, the word 'bank' has one vector. In BERT, it has thousands. How does predicting the context around a [MASK] token enable a model to learn that 'river bank' and 'bank deposit' are different concepts? Does the model learn semantics, or just statistical coincidences?"*

---

## 1 · The Supervised Bottleneck & The Self-Supervised Revolution

### The Old Way: Task-Specific Supervised Learning
Until ~2018, NLP models (LSTMs, early Transformers) were mostly trained from scratch on task-specific labeled data. 
- **Problem:** Labeled data is scarce and expensive ($$$).
- **Consequence:** Models were shallow and struggled with rare words or complex syntax.

### The New Way: Scale via Self-Supervision
The breakthrough was realizing that the **text itself is the label**. By hiding parts of a sentence and asking the model to predict them, we can use the entire internet as a training set.

**The Paradigm Shift:**
1. **Pre-training:** Train a massive model on 100B+ tokens of unlabeled text (General Knowledge).
2. **Fine-tuning:** Adapt that model to a specific task (Sentiment, QA) with a tiny amount of labeled data (Specialization).

| Feature | Static Embeddings (Word2Vec) | Modern Pre-training (BERT/GPT) |
|---------|-----------------------------|-------------------------------|
| **Context** | Context-independent (1 vector per word) | Contextualized (vector depends on neighbors) |
| **Training** | Shallow (Single layer) | Deep (12-96+ layers) |
| **Transfer** | Just the embeddings | Entire model weights |
| **Polysemy** | Struggles with 'bank', 'crane' | Solves by reading context |

---
## 2 · The Architectural Trinity

Transformers are like LEGO blocks; how you stack them determines what they learn.

### A. Encoder-only (Auto-encoding)
- **Examples:** BERT, RoBERTa, ALBERT.
- **Attention:** Bidirectional (sees left AND right context).
- **Best For:** Natural Language Understanding (NLU) — Classification, NER, Sentiment.

### B. Decoder-only (Auto-regressive)
- **Examples:** GPT series, Llama, Mistral.
- **Attention:** Causal (masked) — only sees past tokens.
- **Best For:** Natural Language Generation (NLG) — Chatbots, Creative writing.

### C. Encoder-Decoder (Seq2Seq)
- **Examples:** T5, BART.
- **Attention:** Bidirectional encoder + Causal decoder.
- **Best For:** Conditional Generation — Translation, Summarization.

---

## 3 · Deep Dive: BERT (Bidirectional Encoder Representations from Transformers)

BERT was the first model to successfully use a deep bidirectional encoder for pre-training.

### Objective 1: Masked Language Modeling (MLM)
BERT hides 15% of tokens. To prevent the model from only learning to see the `[MASK]` token, it uses the **80/10/10 Rule**:
- **80%** of the time: Replace with `[MASK]`.
- **10%** of the time: Replace with a **random word** (forces the model to check if the word fits).
- **10%** of the time: Keep it **unchanged** (forces the model to always build good representations).

### Objective 2: Next Sentence Prediction (NSP)
Model receives two segments (A and B). It must predict if B follows A. 
- *Note: Later models like RoBERTa found that removing NSP actually improved performance on most downstream tasks.*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForMaskedLM.from_pretrained('bert-base-uncased')

def probe_bert(text):
    inputs = tokenizer(text, return_tensors='pt')
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    
    with torch.no_grad():
        logits = model(**inputs).logits
        
    mask_token_logits = logits[0, mask_token_index, :]
    top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
    
    print(f"Text: {text}")
    for i, token in enumerate(top_5_tokens):
        print(f"  Top {i+1}: {tokenizer.decode([token])}")

probe_bert("The capital of France is [MASK].")
probe_bert("The chef prepared a delicious [MASK] for the guests.")

---
## 4 · Deep Dive: GPT-style Pre-training

While BERT looks at context bidirectionally, GPT looks **Forward Only**.

### Causal Language Modeling (CLM)
The objective is simple: Predict token $t$ given $t-1, t-2, ... 1$. 

**The Mathematical View:**
Maximize the likelihood: $L = \sum_i \log P(x_i | x_{<i}; \theta)$

**Why it's better for generation:**
Since the model never sees the "future" during training, it doesn't cheat. This makes it a natural fit for sampling one word at a time to generate text.

---
## 5 · Post-BERT Optimizations (The Evolution)

After BERT, several models refined the recipe:

1. **RoBERTa (The Robust BERT):**
   - Removed NSP (found it unnecessary).
   - Dynamic Masking: Masking patterns change every epoch (BERT mask was static).
   - 10x more data, 10x longer training.

2. **ALBERT (A Lite BERT):**
   - **Factorized Embedding:** Separate vocabulary size from hidden layer size.
   - **Cross-Layer Parameter Sharing:** All 12 layers share the SAME weights. 90% fewer parameters with similar performance!

3. **ELECTRA (Replaced Token Detection):**
   - Instead of hiding tokens, it replaces some with plausibly wrong words from a small "generator" network.
   - The model acts as a **Discriminator** (Is this word original or fake?).
   - Much more efficient because it learns from EVERY token, not just the 15% masked ones.

---
## 6 · Implementation: Contextualized Embeddings

Let's prove that BERT word vectors change based on context. We will compare the vector for the word **"bank"** across three sentences.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
from torch.nn.functional import cosine_similarity

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

sentences = [
    "I need to go to the bank to deposit my paycheck.",
    "The river bank was muddy after the rain.",
    "You can always bank on her to get the job done."
]

embeddings = []

for sent in sentences:
    inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True)
    # Note: bert-base-uncased tokenizes these 'bank's as a single token [2910]
    # We find the position in the current sentence input_ids
    token_ids = inputs['input_ids'][0].tolist()
    token_idx = token_ids.index(2910)
    
    with torch.no_grad():
        outputs = model(**inputs)
        # Get the last hidden state for the 'bank' token
        # Shape: [batch, seq_len, hidden_dim]
        emb = outputs.last_hidden_state[0, token_idx, :]
        embeddings.append(emb)

print("Cosine Similarity Matrix for 'bank':")
print(f"Financial vs River:    {cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0)).item():.4f}")
print(f"Financial vs Verb:     {cosine_similarity(embeddings[0].unsqueeze(0), embeddings[2].unsqueeze(0)).item():.4f}")
print(f"River vs Verb:         {cosine_similarity(embeddings[1].unsqueeze(0), embeddings[2].unsqueeze(0)).item():.4f}")

print("\nObservation: Even though the token ID is identical (2910), the vectors are distinct!")
print("This is the core power of contextualized embeddings.")

---
## ✅ Knowledge Check

1. **Why does BERT use the 80/10/10 rule?**
   - Because at fine-tuning time, the model never sees `[MASK]`. If it only learned from `[MASK]`, its performance would decouple from real-world text.

2. **Which architecture is best for summarization?**
   - Encoder-Decoder (e.g., T5). You need the encoder to understand the long text and the decoder to generate the short summary.

3. **What is the main limitation of Causal LLMs (GPT)?**
   - They cannot see the "future context". For tasks like NER or translation, seeing the whole sentence at once is often better.

---

## 🔨 Practical Practice

1. **Probing Knowledge:** Use the `AutoModelForMaskedLM` code to see if BERT knows facts like "Barack Obama was the [MASK] of the USA."
2. **Similarity Search:** Take a list of 5 sentences and find which two are most similar using the `[CLS]` token embedding from BERT.

---
**Next →** NLP 06: HuggingFace Pipelines & Transfer Learning